In [ ]:
%pip install ogb
%pip install EoN
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
%pip install torch_geometric



# Config

In [1]:
config = {
    "undirected": False,

    "model": {
        "emb_dim": 256,
        "num_layers": [8],
        "dropout": 0.2,

        "conv_type": ["dir_gcn","dir_gin"],
        "num_heads": 2,
        "alpha": 0.8, # for directed GNNs

        # Multiple pooling strategies
        "pool_types": ["attention"],

        "use_residual": True,
        "use_graph_norm": True,
    },

    "train": {
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-4,
        "weight_decay": 1e-5,
        "grad_clip": 1.0,
    },

    "data": {
        "max_depth": 20,
        "num_workers": 4,
    },

    "exp": {
        "seed": 42,
        "device": "cuda",
    }
}

In [3]:
import torch
import random
import numpy as np
import os

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Seed set to: {seed}")
set_seed(config["exp"]["seed"])

Seed set to: 42


# Data Loading

In [ ]:
from ogb.graphproppred import PygGraphPropPredDataset
from torch_geometric.data import DataLoader
import torch
import torch_geometric
torch.serialization.add_safe_globals([torch_geometric.data.data.DataEdgeAttr, torch_geometric.data.data.DataTensorAttr,torch_geometric.data.storage.GlobalStorage])

from torch_geometric.utils import to_undirected

def make_undirected_transform(data):
    if config["undirected"]:
        data.edge_index = to_undirected(
            data.edge_index,
            num_nodes=data.num_nodes
        )
    return data

d_name= "ogbg-code2"
dataset = PygGraphPropPredDataset(name = d_name, pre_transform=make_undirected_transform)

split_idx = dataset.get_idx_split()
train_loader = DataLoader(dataset[split_idx["train"]], batch_size=config["train"]["batch_size"], shuffle=True)
valid_loader = DataLoader(dataset[split_idx["valid"]], batch_size=config["train"]["batch_size"], shuffle=False)
test_loader = DataLoader(dataset[split_idx["test"]], batch_size=config["train"]["batch_size"], shuffle=False)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import os
node_attr_path = os.path.join(os.curdir,dataset.root,"mapping","attridx2attr.csv.gz")
type_idx_path = os.path.join(os.curdir,dataset.root,"mapping","typeidx2type.csv.gz")

In [ ]:
import os
import pandas as pd

node_attr_mapping  =  pd.read_csv(node_attr_path)
type_idx_mapping = pd.read_csv(type_idx_path)
node_type_vocab_size = len(type_idx_mapping)

In [ ]:
new_rows = pd.DataFrame([{'attr idx': 10030, 'attr': '<pad>'},{'attr idx':10031,'attr':'<sos>'},{'attr idx':10032,'attr':'<eos>'}])
node_attr_mapping = pd.concat([node_attr_mapping, new_rows], ignore_index=True)

In [ ]:
node_attr_to_idx_dict = {}
node_idx_to_attr_dict = {}
for i in range(len(node_attr_mapping)):
  node_attr_to_idx_dict[node_attr_mapping.iloc[i]['attr']] = node_attr_mapping.iloc[i]["attr idx"]
  node_idx_to_attr_dict[node_attr_mapping.iloc[i]["attr idx"]] = node_attr_mapping.iloc[i]["attr"]


# Encoder

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import (
    GINConv, GCNConv, GATv2Conv,DirGNNConv,
    GraphNorm,
    global_add_pool, global_mean_pool, global_max_pool,
    GlobalAttention,
    Set2Set # Added Set2Set import
)
class MultiHeadGlobalAttention(nn.Module):
    def __init__(self, emb_dim, heads=4):
        super().__init__()
        self.heads = heads

        self.gate_nn = nn.ModuleList([
            nn.Sequential(
                nn.Linear(emb_dim, 2*emb_dim),
                nn.LayerNorm(2*emb_dim),
                nn.ReLU(),
                nn.Linear(2*emb_dim, 1)
            )
            for _ in range(heads)
        ])

    def forward(self, x, batch):
        out = []

        for gate in self.gate_nn:
            scores = gate(x)                       # [N, 1]
            weights = torch_geometric.utils.softmax(scores, batch)
            pooled = torch_geometric.nn.global_add_pool(x * weights, batch)
            out.append(pooled)

        return torch.cat(out, dim=-1)

# ---------- Conv Factory ----------


def build_conv(conv_type, emb_dim, heads=2):
    if conv_type == "gin":
        mlp = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim)
        )
        return GINConv(nn=mlp, train_eps=True)

    elif conv_type == "dir_gatv2":
        base_conv = GATv2Conv(
            emb_dim,
            emb_dim ,
            heads,
            concat=False
        )
        return DirGNNConv(base_conv,alpha=config["model"]["alpha"])
        
    elif conv_type == "dir_gin":
        mlp = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim)
        )
        base_conv = GINConv(nn=mlp, train_eps=True)
        base_conv.in_channels = emb_dim
        base_conv.out_channels = emb_dim
        return DirGNNConv(base_conv,alpha=config["model"]["alpha"])
        
    elif conv_type == "dir_gcn":
        base_conv = GCNConv(emb_dim,emb_dim)
        return DirGNNConv(base_conv,alpha=config["model"]["alpha"])

    elif conv_type == "gcn":
        return GCNConv(emb_dim, emb_dim)

    elif conv_type == "gatv2":
        assert emb_dim % heads == 0
        return GATv2Conv(
            emb_dim,
            emb_dim//2,
            heads=heads,
            concat=True
        )

    else:
        raise ValueError(f"Unknown conv type: {conv_type}")

def build_pool(pool_type, emb_dim, use_multi_stat=False, attn_heads=2):

    if pool_type == 'multi_stat':
        def pool(x, batch):
            return torch.cat([
                global_mean_pool(x, batch),
                global_max_pool(x, batch),
                global_add_pool(x, batch)
            ], dim=-1)
        return pool, emb_dim * 3

    if pool_type == "add":
        return lambda x, batch: global_add_pool(x, batch), emb_dim

    elif pool_type == "mean":
        return lambda x, batch: global_mean_pool(x, batch), emb_dim

    elif pool_type == "attention":
        if attn_heads == 1:
            gate_nn = nn.Sequential(
                nn.Linear(emb_dim, 2*emb_dim),
                nn.LayerNorm(2*emb_dim),
                nn.ReLU(),
                nn.Linear(2*emb_dim, 1)
            )
            return GlobalAttention(gate_nn), emb_dim
        else:
            pool = MultiHeadGlobalAttention(emb_dim, heads=attn_heads)
            return pool, emb_dim * attn_heads

    elif pool_type == "set2set":
        pool = Set2Set(emb_dim, processing_steps=3)
        return pool, emb_dim * 2

    else:
        raise ValueError(f"Unknown pool type: {pool_type}")

# ---------- Encoder ----------
class Encoder(nn.Module):
    def __init__(
        self,
        type_id_vocab_size,
        attr_id_vocab_size,
        max_depth,
        emb_dim,
        num_layers=3,
        conv_type="gin",
        pool_type="mean",
        dropout=0.2,
        num_heads=2,
        use_virtual_node=False,
        use_depth = True,
        use_residual = True,
        use_graph_norm = True
    ):
        super().__init__()

        self.emb_dim = emb_dim
        self.max_depth = max_depth
        self.num_layers = num_layers
        self.dropout = dropout
        self.use_virtual_node = use_virtual_node
        self.use_depth = use_depth
        self.use_residual = use_residual
        self.use_graph_norm = use_graph_norm

        # ---- Embeddings ----
        self.type_emb = nn.Embedding(type_id_vocab_size, emb_dim)
        self.attr_emb = nn.Embedding(attr_id_vocab_size, emb_dim)
        if use_depth:
          self.depth_emb = nn.Embedding(max_depth + 1, emb_dim)

          self.input_proj = nn.Linear(3 * emb_dim, emb_dim)
        else:
          self.input_proj = nn.Linear(2 * emb_dim, emb_dim)

        # ---- Conv stack ----
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(num_layers):
            self.convs.append(build_conv(conv_type, emb_dim, num_heads))
            self.norms.append(GraphNorm(emb_dim))

        # ---- Virtual Node ----
        if use_virtual_node:
            self.virtual_node_emb = nn.Embedding(1, emb_dim)
            self.virtual_mlp = nn.Sequential(
                nn.Linear(emb_dim, emb_dim),
                nn.ReLU(),
                nn.Linear(emb_dim, emb_dim)
            )

        # ---- Pooling ----
        self.pool, self.pool_dim = build_pool(
            pool_type, emb_dim
        )
        self.fc_out = nn.Sequential(nn.Linear(self.pool_dim,emb_dim*2),nn.ReLU(),nn.LayerNorm(emb_dim*2),nn.Linear(emb_dim*2,emb_dim),nn.LayerNorm(emb_dim))

    def forward(self, x, edge_index, depth, batch):
        # ---- Unpack ----
        type_ids = x[:, 0]
        attr_ids = x[:, 1]
        node_depth = torch.clamp(depth.squeeze(-1), 0, self.max_depth)

        # ---- Embed ----
        x1 = self.type_emb(type_ids)
        x2 = self.attr_emb(attr_ids)
        if self.use_depth:
          x3 = self.depth_emb(node_depth)
          x = torch.cat([x1, x2, x3], dim=1)
        else:
          x = torch.cat([x1, x2], dim=1)
        x = self.input_proj(x)

        # ---- Init virtual node ----
        if self.use_virtual_node:
            num_graphs = batch.max().item() + 1
            virtual_node = self.virtual_node_emb.weight.repeat(num_graphs, 1)

        # ---- Message Passing ----
        for layer, (conv, norm) in enumerate(zip(self.convs, self.norms)):

            # Add virtual node to all nodes
            if self.use_virtual_node:
                x = x + virtual_node[batch]

            h = conv(x, edge_index)
            h = F.elu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)

            if self.use_residual:
                x = x + h
            if self.use_graph_norm:
                x = norm(x)

            # Update virtual node
            if self.use_virtual_node:
                pooled = global_add_pool(x, batch)
                virtual_node = virtual_node + self.virtual_mlp(pooled)

        # ---- Pooling ----
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.pool(x, batch)
        x = self.fc_out(x)

        return x

# Decoder

In [ ]:
import torch
import torch.nn as nn
import math

class GraphConditionedTransformerDecoder(nn.Module):
    def __init__(
        self,
        embedding,
        vocab_size,
        emb_dim,
        pad_token_id,
        num_layers=4,
        nhead=4,
        max_len=20,
        dropout=0.1
    ):
        super().__init__()

        self.emb_dim = emb_dim
        self.embedding = embedding
        self.pad_token_id = pad_token_id

        # Positional embedding
        self.pos_emb = nn.Embedding(max_len, emb_dim)

        self.graph_ln = nn.LayerNorm(emb_dim)

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=nhead,
            batch_first=True,
            dropout=dropout
        )

        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers)

        # Output projection (weight tying, no bias)
        self.fc_out = nn.Linear(emb_dim, vocab_size, bias=False)
        self.fc_out.weight = self.embedding.weight

    def forward(self, graph_emb, tgt):
        """
        graph_emb: [B, H]
        tgt: [B, T]
        """

        B, T = tgt.shape

        # Token embeddings
        tok_emb = self.embedding(tgt) * math.sqrt(self.emb_dim)

        pos = torch.arange(T, device=tgt.device).unsqueeze(0)
        pos_emb = self.pos_emb(pos)

        x = tok_emb + pos_emb  # [B, T, H]

        graph_token = graph_emb.unsqueeze(1)  # [B, 1, H] 

        x = torch.cat([graph_token, x], dim=1)  # [B, T+1, H]

        seq_len = x.size(1)

        mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device),
            diagonal=1
        )
        mask = mask.masked_fill(mask == 1, float('-inf')).masked_fill(mask == 0, 0.0)

        mask[:, 0] = 0

        padding_mask = (tgt == self.pad_token_id)  # [B, T]

        graph_pad = torch.zeros((B, 1), dtype=torch.bool, device=tgt.device)

        padding_mask = torch.cat([graph_pad, padding_mask], dim=1)  # [B, T+1]

        x = self.transformer(
            x,
            mask=mask,
            src_key_padding_mask=padding_mask
        )

        # Remove graph token
        x = x[:, 1:, :]  # [B, T, H]

        logits = self.fc_out(x)

        return logits

# Graph2Seq

In [ ]:
import torch
import torch.nn as nn

class Graph2SeqModel(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, data, tgt):
        graph_emb = self.encoder(
            data.x.to(device),
            data.edge_index.to(device),
            data.node_depth.to(device),
            data.batch.to(device)
        )

        logits = self.decoder(graph_emb, tgt)

        return logits

    def generate(self, data, max_tokens):
        graph_emb = self.encoder(
            data.x.to(device),
            data.edge_index.to(device),
            data.node_depth.to(device),
            data.batch.to(device)
        )

        num_graphs_in_batch = data.num_graphs

        tgt = torch.tensor([[10031] for i in range(num_graphs_in_batch)], device=device)

        for i in range(max_tokens):
            logits = self.decoder(graph_emb, tgt)
            next_token = torch.argmax(logits[:, -1, :], dim=1)
            tgt = torch.cat([tgt, next_token.unsqueeze(-1)], dim=1)

        return tgt

# Eval code

In [ ]:
from ogb.graphproppred import Evaluator

evaluator = Evaluator(name = "ogbg-code2")

def evaluate_f1(data_loader):
    model.eval()

    seq_ref = []
    seq_pred = []

    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)

            y = batch.y

            # Model predictions
            pred = model.generate(batch, 10)

            for i in range(len(pred)):
                pred_tokens = []
                for j in range(len(pred[i])):
                    pred_tokens.append(
                        node_idx_to_attr_dict[pred[i][j].item()]
                    )
                    if(pred[i][j].item()==EOS):
                        break
                if "<sos>" in pred_tokens:
                  pred_tokens.remove("<sos>")
                if "<eos>" in pred_tokens:
                  pred_tokens.remove("<eos>")
                seq_pred.append(pred_tokens)

            for i in range(len(y)):

                seq_ref.append(y[i])

    input_dict = {
        "seq_ref": seq_ref,
        "seq_pred": seq_pred
    }

    result = evaluator.eval(input_dict)
    return result

# Train code

In [ ]:
import torch
from torch.optim import Adam
from torch.nn import CrossEntropyLoss

# Token IDs
PAD = 10030
SOS = 10031
EOS = 10032
UNK = 10029


def add_padding(batch_y, device):
    sequences = []

    # Build sequences with <sos> and <eos>
    for seq in batch_y:
        current_seq = [SOS]

        for x in seq:
            current_seq.append(node_attr_to_idx_dict.get(x, UNK))

        current_seq.append(EOS)
        sequences.append(current_seq)

    max_len = max(len(seq) for seq in sequences)

    # Pad
    padded = [
        seq + [PAD] * (max_len - len(seq))
        for seq in sequences
    ]

    return torch.tensor(padded, dtype=torch.long, device=device)


def train(model, train_loader,valid_loader,device,model_path):
    model.to(device)

    criterion = CrossEntropyLoss(ignore_index=PAD)
    optimizer = Adam(model.parameters(), lr=config["train"]["lr"])
    val_f1_score=0
    limit = 5
    for epoch in range(config["train"]["epochs"]):
        model.train()
        total_loss = 0
        total_tokens = 0

        for batch in train_loader:
            batch = batch.to(device)

            y_padded = add_padding(batch.y, device)

            decoder_input = y_padded[:, :-1]      # <sos> A B C
            target_for_loss = y_padded[:, 1:]     # A B C <eos>

            optimizer.zero_grad()

            logits = model(batch, decoder_input)

            # Compute loss
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                target_for_loss.reshape(-1)
            )

            loss.backward()


            torch.nn.utils.clip_grad_norm_(model.parameters(), config["train"]["grad_clip"])

            optimizer.step()

            non_pad = (target_for_loss != PAD).sum().item()
            total_loss += loss.item() * non_pad
            total_tokens += non_pad

        avg_loss = total_loss / total_tokens
        if val_f1_score<float(evaluate_f1(valid_loader)["F1"]):
            torch.save(model.state_dict(),model_path)
            limit=5
        else:
            limit-=1
            print(f"Epoch {epoch+1} saw a decrease in val score, will try atmost {limit} more epochs to see in case val score increases")
            if(limit==0):
                break
        print(f"Epoch {epoch+1}, Loss: {avg_loss:.6f}")




# Training Loop

In [ ]:
for layers in config["model"]["num_layers"]:
  for conv_type in config["model"]["conv_type"]:
      for pool_type in config["model"]["pool_types"]:

          encoder = Encoder(
              type_id_vocab_size=node_type_vocab_size,
              attr_id_vocab_size=node_attr_mapping.shape[0],
              max_depth=config["data"]["max_depth"],
              emb_dim=config["model"]["emb_dim"],
              num_layers=layers,
              conv_type=conv_type,
              pool_type=pool_type,
              dropout=config["model"]["dropout"],
              num_heads=config["model"]["num_heads"],
              use_residual = config["model"]["use_residual"],
              use_graph_norm = config["model"]["use_graph_norm"]
          ).to(device)

          decoder =  GraphConditionedTransformerDecoder(encoder.attr_emb, node_attr_mapping.shape[0], config["model"]["emb_dim"], 10030).to(device)
          model = Graph2SeqModel(encoder,decoder).to(device)
          train(model,train_loader,valid_loader,device,f'{conv_type+"_"+pool_type}.pt')
          model.load_state_dict(torch.load(f'{conv_type+"_"+pool_type}.pt'))
          val_score = evaluate_f1(valid_loader)
          test_score = evaluate_f1(test_loader)

          print(f'{conv_type+"_"+pool_type+"_"}{layers}')
          print("============")
          print(val_score)
          print("===========")
          print(test_score)

          with open(f'{conv_type}_{pool_type}_{layers}.eval',"w") as fh:

              fh.write("Validation Score\n=====\n")
              fh.write(f'{val_score}\n\n')
              fh.write("Test Score\n=====\n")
              fh.write(f'{test_score}')